Backtracking 8 PUZZLE

Phạm Vĩ Cận - 24110169

git: https://github.com/SuperDracula/AI-class/blob/main/Back_8PV.ipynb

In [3]:
import pygame
import sys
import time
import random

def get_actions(matrix):
    for i in range(3):
        for j in range(3):
            if matrix[i][j] == 0:
                r, c = i, j
                possible_moves = []
                if r > 0: possible_moves.append("UP")
                if r < 2: possible_moves.append("DOWN")
                if c > 0: possible_moves.append("LEFT")
                if c < 2: possible_moves.append("RIGHT")
                return r, c, possible_moves
    return -1, -1, []

def apply_move(state, move):
    r, c = -1, -1
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                r, c = i, j
                
    new_state = [row[:] for row in state]
    if move == "LEFT" and c > 0:
        new_state[r][c], new_state[r][c - 1] = new_state[r][c - 1], new_state[r][c]
    elif move == "RIGHT" and c < 2:
        new_state[r][c], new_state[r][c + 1] = new_state[r][c + 1], new_state[r][c]
    elif move == "UP" and r > 0:
        new_state[r][c], new_state[r - 1][c] = new_state[r - 1][c], new_state[r][c]
    elif move == "DOWN" and r < 2:
        new_state[r][c], new_state[r + 1][c] = new_state[r + 1][c], new_state[r][c]
    return new_state

def is_goal(state):
    return state == [[1, 2, 3],
                     [4, 5, 6],
                     [7, 8, 0]]

def to_tuple(matrix):
    return tuple(tuple(row) for row in matrix)

def recursive_backtrack(current_state, path, visited_in_path, depth, limit):
    if is_goal(current_state):
        return path
        
    if depth >= limit:
        return None
        
    _, _, moves = get_actions(current_state)
    
    random.shuffle(moves)
    
    for move in moves:
        next_state = apply_move(current_state, move)
        next_tuple = to_tuple(next_state)
        
        if next_tuple not in visited_in_path:
            
            visited_in_path.add(next_tuple)
            flat_next = [val for row in next_state for val in row]
            path.append({'matrix': flat_next, 'move': move})
            
            result = recursive_backtrack(next_state, path, visited_in_path, depth + 1, limit)
            
            if result is not None:
                return result 
                
            path.pop()
            visited_in_path.remove(next_tuple)
            
    return None 

def do_Backtracking(start_state, max_depth=40):
    flat_start = [val for row in start_state for val in row]
    initial_path = [{'matrix': flat_start, 'move': "START"}]
    
    visited_in_path = set()
    visited_in_path.add(to_tuple(start_state))
    
    return recursive_backtrack(start_state, initial_path, visited_in_path, 0, max_depth)


BACKGROUND_COLOR = (0, 0, 0)
TILE_COLOR = (52, 152, 219) 
BTN_COLOR = (46, 204, 113)  
BACK_BTN_COLOR = (231, 76, 60) 
TEXT_COLOR = (255, 255, 255)      
LIGHT_TEXT = (220, 220, 220)       
LINE_COLOR = (80, 80, 80)          

TILE_SIZE = 120
MARGIN = 8

GAME_WIDTH = TILE_SIZE * 3 + MARGIN * 4 
LOG_PANEL_WIDTH = 280 
BOTTOM_PANEL = 80

TOTAL_WIDTH = GAME_WIDTH + LOG_PANEL_WIDTH
TOTAL_HEIGHT = GAME_WIDTH + BOTTOM_PANEL

def get_grid_pos(index):
    return index // 3, index % 3

def get_pixel_coords(row, col):
    x = MARGIN + col * (TILE_SIZE + MARGIN)
    y = MARGIN + row * (TILE_SIZE + MARGIN)
    return x, y

def draw_static_state(screen, font, state, ignore_idx=-1, solution_path=None, current_step_idx=0, scroll_line=None):
    screen.fill(BACKGROUND_COLOR)

    for i, val in enumerate(state):
        if val != 0 and i != ignore_idx:
            row, col = get_grid_pos(i)
            x, y = get_pixel_coords(row, col)
            pygame.draw.rect(screen, TILE_COLOR, (x, y, TILE_SIZE, TILE_SIZE), border_radius=8)
            text_surface = font.render(str(val), True, TEXT_COLOR)
            text_rect = text_surface.get_rect(center=(x + TILE_SIZE // 2, y + TILE_SIZE // 2))
            screen.blit(text_surface, text_rect)

    btn_font = pygame.font.Font(None, 30)
    back_btn_rect = pygame.Rect(GAME_WIDTH - 150, GAME_WIDTH + 15, 140, 45)
    pygame.draw.rect(screen, BACK_BTN_COLOR, back_btn_rect, border_radius=8)
    
    btn_surf = btn_font.render("Quay lai Menu", True, TEXT_COLOR)
    btn_rect = btn_surf.get_rect(center=back_btn_rect.center)
    screen.blit(btn_surf, btn_rect)
    
    pygame.draw.line(screen, LINE_COLOR, (GAME_WIDTH, 0), (GAME_WIDTH, TOTAL_HEIGHT), 3)
    
    log_font = pygame.font.Font(None, 24) 
    log_x = GAME_WIDTH + 15
    log_y = 15
    line_height = 20 
    
    title = pygame.font.Font(None, 35).render("LICH SU QUA TRINH", True, LIGHT_TEXT) 
    screen.blit(title, (log_x, log_y))
    
    current_scroll = 0
    max_scroll = 0

    if solution_path is not None:
        all_lines = []
        for i in range(current_step_idx + 1):
            item = solution_path[i]
            all_lines.append(f"--- Buoc {i} | Lenh: {item['move']} ---")
            r = item['matrix']
            all_lines.append(f"   [{r[0]}]  [{r[1]}]  [{r[2]}]")
            all_lines.append(f"   [{r[3]}]  [{r[4]}]  [{r[5]}]")
            all_lines.append(f"   [{r[6]}]  [{r[7]}]  [{r[8]}]")
            all_lines.append("") 
            
        max_visible_lines = (TOTAL_HEIGHT - 60) // line_height
        max_scroll = max(0, len(all_lines) - max_visible_lines)

        if scroll_line is None:
            current_scroll = max_scroll 
        else:
            current_scroll = max(0, min(scroll_line, max_scroll))
            
        visible_lines = all_lines[current_scroll : current_scroll + max_visible_lines]
        
        for idx, line_text in enumerate(visible_lines):
            if line_text != "":
                surf = log_font.render(line_text, True, LIGHT_TEXT)
                screen.blit(surf, (log_x, log_y + 40 + idx * line_height))
                
    return back_btn_rect, current_scroll, max_scroll

def animate_transition(screen, font, clock, curr_item, nxt_item, solution_path, step_num, scroll_line, animation_steps=25):
    current_state = curr_item['matrix']
    next_state = nxt_item['matrix']
    
    curr_zero_idx = current_state.index(0)
    next_zero_idx = next_state.index(0)
    moving_tile_val = current_state[next_zero_idx]
    
    old_row, old_col = get_grid_pos(next_zero_idx)
    new_row, new_col = get_grid_pos(curr_zero_idx)
    old_x, old_y = get_pixel_coords(old_row, old_col)
    new_x, new_y = get_pixel_coords(new_row, new_col)
    
    for step in range(animation_steps + 1):
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                return False
                
        progress = step / animation_steps
        current_x = old_x + (new_x - old_x) * progress
        current_y = old_y + (new_y - old_y) * progress
        
        draw_static_state(screen, font, current_state, ignore_idx=next_zero_idx, 
                          solution_path=solution_path, current_step_idx=step_num, scroll_line=scroll_line)
        
        pygame.draw.rect(screen, TILE_COLOR, (current_x, current_y, TILE_SIZE, TILE_SIZE), border_radius=8)
        text_surface = font.render(str(moving_tile_val), True, TEXT_COLOR)
        text_rect = text_surface.get_rect(center=(current_x + TILE_SIZE // 2, current_y + TILE_SIZE // 2))
        screen.blit(text_surface, text_rect)
        
        pygame.display.flip()
        clock.tick(60)
    return True

def show_main_menu(screen):
    title_font = pygame.font.Font(None, 40)
    btn_font = pygame.font.Font(None, 35)
    
    btn_width, btn_height = 300, 60
    center_x = TOTAL_WIDTH // 2 - btn_width // 2
    
    btn_rect = pygame.Rect(center_x, 150, btn_width, btn_height)
    
    while True:
        screen.fill(BACKGROUND_COLOR)
        
        title_surf = title_font.render("BACKTRACKING (CSP) AGENT", True, LIGHT_TEXT)
        title_rect = title_surf.get_rect(center=(TOTAL_WIDTH // 2, 70))
        screen.blit(title_surf, title_rect)
        
        pygame.draw.rect(screen, BTN_COLOR, btn_rect, border_radius=10)
        btn_surf = btn_font.render("Bat dau", True, TEXT_COLOR)
        btn_text_rect = btn_surf.get_rect(center=btn_rect.center)
        screen.blit(btn_surf, btn_text_rect)
        
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                return False
            if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1: 
                if btn_rect.collidepoint(event.pos):
                    time.sleep(0.1)
                    return True 
        pygame.display.flip()

def show_loading_screen(screen):
    screen.fill(BACKGROUND_COLOR)
    font = pygame.font.Font(None, 35)
    text_surf = font.render("Dang quay lui (Backtracking)...", True, LIGHT_TEXT)
    text_rect = text_surf.get_rect(center=(TOTAL_WIDTH // 2, TOTAL_HEIGHT // 2))
    screen.blit(text_surf, text_rect)
    pygame.display.flip()

def run_visualizer(screen, solution_path):
    font = pygame.font.Font(None, 60)
    clock = pygame.time.Clock()
    
    current_step_idx = 0
    curr_item = solution_path[current_step_idx]
    
    scroll_line = None
    auto_scroll = True
    max_scroll = 0
    
    back_btn_rect, scroll_line_out, max_scroll = draw_static_state(
        screen, font, curr_item['matrix'], 
        solution_path=solution_path, current_step_idx=0, scroll_line=scroll_line
    )
    pygame.display.flip()
    time.sleep(1.5) 
    
    running = True
    is_finished = False
    
    while running:
        if is_finished:
            back_btn_rect, scroll_line_out, max_scroll = draw_static_state(
                screen, font, solution_path[-1]['matrix'],
                solution_path=solution_path, current_step_idx=len(solution_path)-1,
                scroll_line=scroll_line
            )
            pygame.display.flip()
            clock.tick(30)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                return False
            if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
                if back_btn_rect.collidepoint(event.pos):
                    return True

            if event.type == pygame.MOUSEWHEEL:
                if scroll_line is None:
                    scroll_line = max_scroll
                scroll_line -= event.y * 3 
                auto_scroll = False
                
        if not is_finished and current_step_idx < len(solution_path) - 1:
            curr_item = solution_path[current_step_idx]
            nxt_item = solution_path[current_step_idx + 1]
            step_number = current_step_idx + 1
            
            if auto_scroll:
                scroll_line = None

            if not animate_transition(screen, font, clock, curr_item, nxt_item, solution_path, step_number, scroll_line):
                return False
            
            current_step_idx += 1
            time.sleep(0.3) 
        elif not is_finished:
            is_finished = True
    return True

if __name__ == "__main__":
    sys.setrecursionlimit(2000) 
    pygame.init()
    screen = pygame.display.set_mode((TOTAL_WIDTH, TOTAL_HEIGHT))
    pygame.display.set_caption("8-Puzzle Backtracking Agent")
    
    while True:
        start_game = show_main_menu(screen)

        if not start_game:
            break

        show_loading_screen(screen)

        start_state = [[1, 2, 3],
                       [4, 0, 6],
                       [7, 5, 8]]
        
        solution_path = do_Backtracking(start_state, max_depth=40)
                
        if solution_path is not None:
            keep_playing = run_visualizer(screen, solution_path) 
            if not keep_playing:
                break
        else:
            screen.fill(BACKGROUND_COLOR)
            font = pygame.font.Font(None, 40)
            text_surf = font.render("Khong the giai (Vuot Depth Limit)!", True, BACK_BTN_COLOR)
            text_rect = text_surf.get_rect(center=(TOTAL_WIDTH // 2, TOTAL_HEIGHT // 2))
            screen.blit(text_surf, text_rect)
            pygame.display.flip()
            time.sleep(2.5)
            
    pygame.quit()
    print("Da thoat chuong trinh.")

Da thoat chuong trinh.
